# Google Speech Commands

Load Google Speech Commands v0.02 via HuggingFace, confirm 105,829 clips at 16 kHz,
verify the 10-command + unknown + silence class split. Report class distribution (some commands may have
slightly more examples). Listen to 10 samples per class to verify audio quality.

# Overview
This notebook loads and explores the Google Speech Commands v0.02 dataset for keyword spotting. The goal is to build a model that can detect 10 target keywords in the presence of background noise and unknown words.

## Dataset
**Google Speech Commands v0.02**: 105,829 one-second audio clips at 16kHz 
- 10 target keywords: yes, no, up, down, left, right, on, off, stop, go
- Unknown class: all other spoken words (~54,000 clips)
- Silence class: background noise recordings chunked into 1s clips (~400 clips)

**MUSAN**: Music, Speech, and Noise dataset used for noise injection during training
- Makes the model robust to real-world background noise
- Applied as augmentation — clips are still labeled as their original keyword

In [ ]:
import os
import random
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

import torch
# import torchcodec
import torchaudio
from torchaudio.datasets import SPEECHCOMMANDS
from IPython.display import Audio, display

from datasets import load_dataset

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

# I think either of these paths should work, but the second one allows for an environment variable to be set for flexibility.
#DATA_ROOT = "./data"
DATA_ROOT = os.environ.get("DATA_ROOT", "./data")

os.makedirs(DATA_ROOT, exist_ok=True)

In [ ]:
# Import train, val, test
train_set = SPEECHCOMMANDS(DATA_ROOT, download=True, subset="training")
val_set = SPEECHCOMMANDS(DATA_ROOT, download=True, subset="validation")
test_set = SPEECHCOMMANDS(DATA_ROOT, download=True, subset="testing")

print(f"Train: {len(train_set)} \nVal: {len(val_set)} \nTest: {len(test_set)}")
print(f"Total: {len(train_set) + len(val_set)+len(test_set)}")

In [ ]:
# setup silence
silence_dir = os.path.join(DATA_ROOT, "SpeechCommands", "speech_commands_v0.02", "_background_noise_")

SAMPLE_RATE = 16000
CLIP_LENGTH = SAMPLE_RATE * 1

In [ ]:
# import / chunk silence clips to 1 second each
def create_silence_clips(silence_dir, clip_length = CLIP_LENGTH, sr = SAMPLE_RATE):
    silence_clips = []

    for fname in sorted(os.listdir(silence_dir)):
        if not fname.endswith(".wav"):
            continue
        filepath = os.path.join(silence_dir, fname)
        waveform, file_sr = torchaudio.load(filepath)

        if file_sr != sr:
            resampler = torchaudio.transforms.Resample(file_sr, sr)
            waveform = resampler(waveform)

        file_name = fname.replace(".wav", "", )

        num_samples = waveform.shape[1]
        num_clips = num_samples // clip_length

        for i in range(num_clips):
            start = i * clip_length
            end = start + clip_length
            clip = waveform[:, start:end]
            silence_clips.append((clip, sr, "silence", file_name))
        
        print(f"{fname}: {num_clips} clips from {num_samples/sr:.1f}s of audio")

    print(f"\nTotal silence clips: {len(silence_clips)}")
    return silence_clips

silence_clips = create_silence_clips(silence_dir)
for i in random.sample(range(len(silence_clips)), 5):
     clip, sr, label, file_name = silence_clips[i]
     print(f"Label: {label} | File Name: {file_name}")
     display(Audio(clip.numpy(), rate = sr))

##### Inspecting first example tuple structure

In [ ]:
waveform, sample_rate, label, speaker_id, utterance_number = train_set[0]

print(f"Waveform shape: {waveform.shape}")
print(f"Sample rate: {sample_rate} Hz")
print(f"Label: '{label}'")
print(f"Speaker ID: '{speaker_id}'")
print(f"Utterance Number: {utterance_number}")
print(f"Duration: {waveform.shape[1] / sample_rate:.3f}s")


All samples verified at 16000 Hz

In [ ]:
sr_df = pd.DataFrame(
    [train_set[i][1] for i in range(min(500, len(train_set)))],
    columns=["sample_rate"]
)

assert sr_df["sample_rate"].nunique() == 1, "Not all sample rates are equal!"
print(f"All samples verified at {sr_df['sample_rate'].iloc[0]} Hz")

##### Check Labels (Pre-Unknown)

In [ ]:
labels_df = pd.DataFrame([train_set[i][2] for i in range(len(train_set))], columns=["label"])

unique_labels = set(labels_df["label"])

print(sorted(unique_labels))

In [ ]:
# pre-transform value counts (sampled)
value_counts = pd.DataFrame(labels_df.value_counts()).reset_index()
value_counts.head(10)

In [ ]:
# pre-transform class distribution
fig, ax = plt.subplots(figsize = (10,6))
ax.bar(value_counts['label'], value_counts['count'])
ax.set_xlabel("Command")
ax.set_ylabel('frequency')
plt.xticks(rotation=45, ha='right')
ax.set_title("Class distribution")
ax.grid()
plt.show()

##### Transform target and unknown classes

In [ ]:
target_labels = {"yes", "no", "up", "down", "left", "right", "on", "off", "stop", "go"}

def relabel(label):
    if label in target_labels:
        return label
    return "unknown"

relabeled = [relabel(train_set[i][2]) for i in range(len(train_set))]
relabel_df = pd.DataFrame(relabeled, columns=["label"])
relabeled_value_counts = pd.DataFrame(relabel_df.value_counts()).reset_index()
relabeled_value_counts.head(10)

In [ ]:
# post-transform class distribution
fig, ax = plt.subplots(figsize = (10,6))
ax.bar(relabeled_value_counts['label'], relabeled_value_counts['count'])
ax.set_xlabel("Command")
ax.set_ylabel('frequency')
plt.xticks(rotation=45, ha='right')
ax.set_title("Class distribution")
ax.grid()
plt.show()

In [ ]:
for i in random.sample(range(len(train_set)), 5):
    waveform, sample_rate, label, speaker_id, utterance_number = train_set[i]
    print(f"Label: {label} | Speaker: {speaker_id} | Utterance: {utterance_number}")
    display(Audio(waveform.numpy(), rate=sample_rate))

# MUSAN Dataset

Load the MUSAN data via Hugging Face. Verfiy the noise samples are at 16kHz, aligns with the Google Speech Commands dataset's sample rate.

In [ ]:
# Load MUSAN dataset
ds = load_dataset("Aynursusuz/musan-audio-dataset")

In [ ]:
print(f"Splits available: {list(ds.keys())}")
print(f"Train: {len(ds['train'])}")
print(f"\nFeatures: {ds['train'].features}")

In [ ]:
# Inspect a single sample
sample = ds["train"][0]

print(f"Keys: {list(sample.keys())}")

# Accessing the label value and its corresponding name
label_value = sample['label']
label_name = ds["train"].features["label"].names[label_value]

print(f"Label Value: {label_value}")
print(f"Label Name: '{label_name}'")
print(f"Sample rate: {sample['audio']['sampling_rate']} Hz")

waveform = torch.tensor(sample["audio"]["array"]).unsqueeze(0).float()
sample_rate = sample["audio"]["sampling_rate"]

print(f"Waveform shape: {waveform.shape}")
print(f"Duration: {waveform.shape[1] / sample_rate:.3f}s")

In [ ]:
# Sample rate distribution
sr_df = pd.DataFrame(
    [ds["train"][i]["audio"]["sampling_rate"] for i in range(min(200, len(ds["train"])))],
    columns=["sample_rate"]
)

assert sr_df["sample_rate"].nunique() == 1, "Not all MUSAN sample rates are equal!"
print(f"All MUSAN samples verified at {sr_df['sample_rate'].iloc[0]} Hz")

In [ ]:
# Category distribution
category_df = pd.DataFrame(
    [ds["train"].features["label"].names[ds["train"][i]["label"]] for i in range(len(ds["train"]))],
    columns=["category"]
)

unique_categories = sorted(set(category_df["category"]))
print(f"Unique categories: {unique_categories}")

value_counts = category_df["category"].value_counts().reset_index()
value_counts.columns = ["category", "count"]

print(value_counts)

In [ ]:
# Category distribution bar chart
fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(value_counts["category"], value_counts["count"])
ax.set_xlabel("Category")
ax.set_ylabel("Count")
ax.set_title("Category Distribution in MUSAN Dataset")
ax.grid(axis='y')
plt.tight_layout()
plt.show()

In [ ]:
# Listen to random samples per category
for cat in unique_categories:
    cat_indices = [i for i in range(len(ds["train"]))
                   if ds["train"].features["label"].names[ds["train"][i]["label"]] == cat]
    sampled = random.sample(cat_indices, min(4, len(cat_indices)))

    print(f"\n{cat.upper()} samples")
    for i in sampled:
        item = ds["train"][i]
        waveform = np.array(item["audio"]["array"])
        sr = item["audio"]["sampling_rate"]
        label = ds["train"].features["label"].names[item["label"]]
        print(f"Category: {label} | Duration: {len(waveform)/sr:.2f}s | Sample rate: {sr} Hz")
        display(Audio(waveform, rate=sr))

## Phase 1 - MUSAN Injection

In [ ]:
TARGET_SR = 16000
TARGET_LEN = 16000

def fit_noise_to_length(noise_waveform, target_len=TARGET_LEN):
    cur_len = noise_waveform.shape[1]

    if cur_len > target_len:
        start = random.randint(0, cur_len - target_len)
        return noise_waveform[:, start:start + target_len]

    if cur_len < target_len:
        repeats = (target_len // cur_len) + 1
        noise_waveform = noise_waveform.repeat(1, repeats)

    return noise_waveform[:, :target_len]

def compute_rms(waveform, eps=1e-8):
    return torch.sqrt(torch.mean(waveform ** 2) + eps)

def mix_at_snr(speech, noise, snr_db):
    speech_rms = compute_rms(speech)
    noise_rms = compute_rms(noise)

    desired_noise_rms = speech_rms / (10 ** (snr_db / 20))
    scale = desired_noise_rms / (noise_rms + 1e-8)

    noisy = speech + scale * noise
    return torch.clamp(noisy, -1.0, 1.0)

def inject_musan_noise(speech_waveform, musan_item, snr_db):
    noise = torch.tensor(musan_item["audio"]["array"]).float().unsqueeze(0)

    if musan_item["audio"]["sampling_rate"] != TARGET_SR:
        resampler = torchaudio.transforms.Resample(
            musan_item["audio"]["sampling_rate"], TARGET_SR
        )
        noise = resampler(noise)

    noise = fit_noise_to_length(noise, TARGET_LEN)
    speech_waveform = fit_noise_to_length(speech_waveform, TARGET_LEN)

    return mix_at_snr(speech_waveform, noise, snr_db)

### Test

In [ ]:
idx = 0
waveform, sr, label, speaker_id, utt = train_set[idx]

musan_item = ds["train"][10]
noisy = inject_musan_noise(waveform, musan_item, snr_db=10)

print(label)
display(Audio(waveform.numpy(), rate=sr))
display(Audio(noisy.numpy(), rate=sr))


In [ ]:
test_words = ["yes", "no"]
# more words can be added, but for demo purposes we'll just do 2 to keep the output manageable
# test_words = ["yes", "no", "up", "down", "left", "right", "on", "off", "stop", "go"] --- IGNORE ---

snr_levels = [20, 10, 0, -5]

for word in test_words:
    print("\n==============================")
    print("Keyword:", word)

    # find indices of this keyword
    indices = [i for i in range(len(train_set)) if train_set[i][2] == word]

    # sample a few examples
    sampled = random.sample(indices, 2)

    for idx in sampled:
        waveform, sr, label, speaker_id, utt = train_set[idx]
        musan_item = ds["train"][random.randint(0, len(ds["train"]) - 1)]

        print(f"\nSpeaker: {speaker_id}")

        display(Audio(waveform.numpy(), rate=sr))

        for snr in snr_levels:
            noisy = inject_musan_noise(waveform, musan_item, snr_db=snr)
            print(f"SNR = {snr} dB")
            display(Audio(noisy.numpy(), rate=sr))

## Exported Dataset

In [ ]:
import csv
from pathlib import Path

label_vocab = ["yes", "no", "up", "down", "left", "right", "on", "off", "stop", "go", "unknown", "silence"]
label_to_idx = {label: i for i, label in enumerate(label_vocab)}

PROCESSED_ROOT = Path(DATA_ROOT) / "processed"
PROCESSED_ROOT.mkdir(parents=True, exist_ok=True)

In [ ]:
def relabel_command(label):
    return label if label in target_labels else "unknown"

def build_musan_cache(musan_ds, target_sr=TARGET_SR):
    musan_cache = []
    
    for i in range(len(musan_ds["train"])):
        item = musan_ds["train"][i]
        waveform = torch.tensor(item["audio"]["array"]).float().unsqueeze(0)
        sr = item["audio"]["sampling_rate"]
        category = musan_ds["train"].features["label"].names[item["label"]]

        if sr != target_sr:
            resampler = torchaudio.transforms.Resample(sr, target_sr)
            waveform = resampler(waveform)

        musan_cache.append({
            "waveform": waveform,
            "category": category
        })

    return musan_cache

print("Caching MUSAN waveforms...")
musan_cache = build_musan_cache(ds)
print(f"Cached {len(musan_cache)} MUSAN samples")

def inject_musan_noise_cached(speech_waveform, musan_waveform, snr_db):
    noise = fit_noise_to_length(musan_waveform, TARGET_LEN)
    speech_waveform = fit_noise_to_length(speech_waveform, TARGET_LEN)
    return mix_at_snr(speech_waveform, noise, snr_db)

def export_dataset_split(
    speech_dataset,
    split_name,
    output_dir,
    musan_cache=None,
    inject_noise=False,
    fixed_snr=None,
    random_snr_choices=(20, 10, 0, -5),
    include_silence=False,
    silence_clips=None,
    seed=42
):
    rng = random.Random(seed)
    output_dir = Path(output_dir)
    audio_dir = output_dir / "audio"
    audio_dir.mkdir(parents=True, exist_ok=True)

    manifest_path = output_dir / "metadata.csv"

    rows = []

    # export spoken examples
    for i in range(len(speech_dataset)):
        waveform, sr, raw_label, speaker_id, utterance_number = speech_dataset[i]
        label = relabel_command(raw_label)

        snr_used = None
        musan_category = None
        musan_idx = None

        if inject_noise and label != "silence":
            musan_idx = rng.randint(0, len(musan_cache) - 1)
            musan_item = musan_cache[musan_idx]
            musan_category = musan_item["category"]

            if fixed_snr is not None:
                snr_used = fixed_snr
            else:
                snr_used = rng.choice(list(random_snr_choices))

            waveform = inject_musan_noise_cached(
                speech_waveform=waveform,
                musan_waveform=musan_item["waveform"],
                snr_db=snr_used
            )

        waveform = fit_noise_to_length(waveform, TARGET_LEN)

        filename = f"{split_name}_{i:06d}.wav"
        filepath = audio_dir / filename
        torchaudio.save(str(filepath), waveform, TARGET_SR)

        rows.append({
            "file_path": str(filepath),
            "split": split_name,
            "label": label,
            "label_idx": label_to_idx[label],
            "raw_label": raw_label,
            "speaker_id": speaker_id,
            "utterance_number": utterance_number,
            "is_silence": 0,
            "musan_category": musan_category,
            "musan_idx": musan_idx,
            "snr_db": snr_used
        })

    # export silence clips if requested
    if include_silence and silence_clips is not None:
        start_idx = len(rows)

        for j, (clip, sr, label, file_name) in enumerate(silence_clips):
            clip = fit_noise_to_length(clip, TARGET_LEN)

            filename = f"{split_name}_silence_{j:06d}.wav"
            filepath = audio_dir / filename
            torchaudio.save(str(filepath), clip, TARGET_SR)

            rows.append({
                "file_path": str(filepath),
                "split": split_name,
                "label": "silence",
                "label_idx": label_to_idx["silence"],
                "raw_label": "silence",
                "speaker_id": file_name,
                "utterance_number": j,
                "is_silence": 1,
                "musan_category": None,
                "musan_idx": None,
                "snr_db": None
            })

    with open(manifest_path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        writer.writeheader()
        writer.writerows(rows)

    print(f"Exported {len(rows)} examples to {output_dir}")
    print(f"Manifest saved to {manifest_path}")

In [ ]:
export_dataset_split(
    speech_dataset=train_set,
    split_name="train_noisy",
    output_dir=PROCESSED_ROOT / "train_noisy",
    musan_cache=musan_cache,
    inject_noise=True,
    fixed_snr=None,
    random_snr_choices=(20, 10, 0, -5),
    include_silence=True,
    silence_clips=silence_clips,
    seed=42
)

In [ ]:
# Per-SNR training splits (train_clean, train_20db, train_10db, train_0db, train_m5db)
export_dataset_split(
    speech_dataset=train_set,
    split_name="train_clean",
    output_dir=PROCESSED_ROOT / "train_clean",
    musan_cache=musan_cache,
    inject_noise=False,
    include_silence=True,
    silence_clips=silence_clips,
    seed=42
)

for snr in [20, 10, 0, -5]:
    folder_name = f"train_{snr}db" if snr >= 0 else "train_m5db"

    export_dataset_split(
        speech_dataset=train_set,
        split_name=folder_name,
        output_dir=PROCESSED_ROOT / folder_name,
        musan_cache=musan_cache,
        inject_noise=True,
        fixed_snr=snr,
        include_silence=True,
        silence_clips=silence_clips,
        seed=42
    )

In [ ]:
export_dataset_split(
    speech_dataset=val_set,
    split_name="val_clean",
    output_dir=PROCESSED_ROOT / "val_clean",
    musan_cache=musan_cache,
    inject_noise=False,
    include_silence=True,
    silence_clips=silence_clips,
    seed=42
)

In [ ]:
for snr in [20, 10, 0, -5]:
    folder_name = f"val_{snr}db" if snr >= 0 else "val_m5db"
    
    export_dataset_split(
        speech_dataset=val_set,
        split_name=folder_name,
        output_dir=PROCESSED_ROOT / folder_name,
        musan_cache=musan_cache,
        inject_noise=True,
        fixed_snr=snr,
        include_silence=True,
        silence_clips=silence_clips,
        seed=42
    )

In [ ]:
export_dataset_split(
    speech_dataset=test_set,
    split_name="test_clean",
    output_dir=PROCESSED_ROOT / "test_clean",
    musan_cache=musan_cache,
    inject_noise=False,
    include_silence=True,
    silence_clips=silence_clips,
    seed=42
)

In [ ]:
for snr in [20, 10, 0, -5]:
    folder_name = f"test_{snr}db" if snr >= 0 else "test_m5db"
    
    export_dataset_split(
        speech_dataset=test_set,
        split_name=folder_name,
        output_dir=PROCESSED_ROOT / folder_name,
        musan_cache=musan_cache,
        inject_noise=True,
        fixed_snr=snr,
        include_silence=True,
        silence_clips=silence_clips,
        seed=42
    )